# Immune Pathway Involvement in Alzheimer's Disease Risk Genes
#### *An independent exploration using public enrichment and network analysis tools*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb/blob/main/AD_Immune_Patfhways_Notebook.ipynb)

> **Before running:** replace `https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb` above, and again in the setup cell in Section 3, with your actual GitHub username and repository name. See Section 3 for why this is the only edit needed.

---
## 1. Objective

1. To compile the set of genome-wide significant AD risk genes reported by a large, recent AD GWAS meta-analysis.
2. To independently test, using standard gene-set enrichment tools, whether this gene set shows statistically significant enrichment for immune-system-related biological processes.
3. To evaluate whether an unbiased, independently-run analysis (using different tools and a different statistical approach than the original authors) converges on the same broad conclusion.

---
## 2. Data source

The gene list (n = 76 unique genes) was compiled from Tables 1 and 2 of:

> Bellenguez, C., Küçükali, F., Jansen, I.E. et al. (2022). New insights into the genetic etiology of Alzheimer's disease and related dementias. *Nature Genetics*, 54, 412–436. https://doi.org/10.1038/s41588-022-01024-z

This study performed a two-stage GWAS meta-analysis of 111,326 clinically diagnosed or proxy AD/related-dementia cases and 677,663 controls, identifying 75 genome-wide significant risk loci (33 previously known, 42 new at the time of publication). The study's own pathway enrichment analysis found that, beyond the expected amyloid and tau pathway signals, gene sets related to lipid metabolism, endocytosis, and immunity (including macrophage and microglial cell activation) were also significantly enriched.

One entry, "IGH gene cluster," was excluded from the working gene list, due to its genomic complexity as a result of known immunoglobulin gene fusion events.

---
## 3. Reproducibility & setup

This notebook depends on two kinds of external files that live in the same GitHub repository as the notebook itself:

- **`data/Genes.csv`** — the compiled 76-gene list, loaded by the code cell below.
- **`results/*.png`** — four screenshots exported from Enrichr and STRING (Section 5), since those are outputs of external web tools rather than something this notebook generates.

**Recommended setup — commit the files, then link to them, rather than uploading anything by hand:**

1. Push `data/Genes.csv` and the four `results/*.png` files to your GitHub repo, in those two folders, alongside this notebook.
2. The gene-list code cell below tries a local relative path first, and automatically falls back to fetching the file directly from GitHub (via a `raw.githubusercontent.com` URL) if that path doesn't exist. That means:
   - **Locally, or in Colab after cloning the repo** (`!git clone ...`) — the local path is found and used.
   - **In Colab opened fresh via the badge above, with no cloning at all** — the fallback fetches the CSV straight from GitHub. No manual upload, ever.
3. The four result images in Section 5 are linked directly by their GitHub raw URL rather than a relative path. A markdown image tag has no equivalent "try local, else fetch" logic, so a raw URL is the one option that renders correctly in all three places this notebook might be viewed: on GitHub itself, in nbviewer, and in Colab — with zero setup.

This is why the only edit this notebook needs before running anywhere is replacing `https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb`: once in the code cell immediately below, and once in each of the four image lines in Section 5 (a single find-and-replace across the file handles all five in one pass).

In [ ]:
import pandas as pd
from pathlib import Path

# --- one-time setup: replace with your own GitHub username, repo name, and branch ---
GH_USERNAME = "YOUR_USERNAME"
GH_REPO = "YOUR_REPO"
GH_BRANCH = "main"

LOCAL_PATH = Path("data/Genes.csv")
RAW_URL = f"https://raw.githubusercontent.com/{GH_USERNAME}/{GH_REPO}/{GH_BRANCH}/data/Genes.csv"

if LOCAL_PATH.exists():
    genes_df = pd.read_csv(LOCAL_PATH)
    print(f"Loaded gene list from local file: {LOCAL_PATH}")
else:
    genes_df = pd.read_csv(RAW_URL)
    print(f"Local file not found — loaded gene list directly from GitHub:\n{RAW_URL}")

genes_df

---
## 4. Methods

### 4.1 Gene set compilation
All genes listed in Table 1 (known loci) and Table 2 (new loci at time of publication) were extracted. Genes with multiple independent genome-wide significant SNPs reported in the source tables (TREM2 ×3, PLCG2 ×2, SORL1 ×2, SLC24A4 ×2, MME ×2) were deduplicated to a single gene entry each, yielding **76 unique gene symbols**.

### 4.2 Pathway enrichment analysis
The 76-gene list was submitted to **Enrichr** (Chen et al., 2013; Kuleshov et al., 2016), an independent, web-based gene set enrichment tool, and tested against three gene-set libraries:
- **GO Biological Process**
- **KEGG**
- **Reactome**

For each library, Enrichr computes enrichment using a Fisher's exact test against a genome-wide background, and reports both a raw p-value and a Benjamini–Hochberg-corrected adjusted p-value (to control the false discovery rate across the many simultaneous term comparisons performed). Terms with adjusted p-value < 0.05 were retained as statistically significant.

Significant terms were then screened for immune relevance using a fixed keyword list — immune, inflammation, microglia, macrophage, phagocytosis, complement, cytokine, leukocyte, TNF — chosen in advance based on the terminology used in the source paper's own pathway discussion, to reduce post-hoc selection bias. Genes underlying immune-flagged terms were pooled across all three libraries (union) and deduplicated, then manually reviewed to confirm that their primary annotated function was immune-related rather than an incidental match to a narrow GO term.

### 4.3 Protein–protein interaction network analysis
The same full 76-gene list (not the immune-flagged subset) was submitted to **STRING v12** (Szklarczyk et al., 2023) to test, independently of the enrichment analysis above, whether immune-relevant genes form a distinguishable, densely interconnected network module based on documented protein–protein interaction evidence (co-expression, curated databases, experimental data, and text-mining).
STRING's built-in functional enrichment output (its own implementation of GO term enrichment, computed via a different backend than Enrichr) was used as a secondary, independent statistical cross-check.

### 4.4 Cross-validation against the source paper
The final immune-flagged gene set was compared against the specific genes the Bellenguez et al. (2022) paper explicitly discussed as immune/microglia-related in its main text (TREM2, RHOH, BLNK, SIGLEC11, LILRB2, SHARPIN, RBCK1, OTULIN, ADAM17, TNIP1, SPPL2A), to assess convergence between this independent analysis and the original authors' own conclusions.

---
## 5. Results

### 5.1 Unbiased enrichment across the full gene set
![GO enrichment - full 76-gene list, top 10 terms by significance](https://raw.githubusercontent.com/https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb/main/results/enrichr_full_76_genes.png)

Across the top 10 most statistically significant GO Biological Process terms for the full, unbiased 76-gene list, two clear biological themes emerged without any pre-selection: **amyloid processing** and **immune system activity** (e.g., immune effector process, myeloid leukocyte activation, positive regulation of immune system process). "Immune effector process" was the single most gene-dense term among the top 10, involving approximately 13 of the 76 input genes.

This is the key independent confirmation this project set out to test: the same two major biological signals reported by Bellenguez et al. — amyloid/tau processing and immune/microglial involvement — emerged from an unbiased, independently-run enrichment analysis.

### 5.2 Immune-flagged gene subset
Combining significant, keyword-flagged terms across GO Biological Process, KEGG, and Reactome, followed by manual review, yielded **26 of 76 genes (34%)** classified as immune-relevant:

CR1, PLCG2, GRN, APP, ADAM17, IL34, SHARPIN, INPP5D, SPI1, CLU, SCIMP, ABCA7, TREM2, ACE, PTK2B, SPPL2A, SORL1, TNIP1, LILRB2, HLA-DQA1, MME, TSPAN14, SEC61G, TREML2, CTSH, CTSB, SIGLEC11, BLNK, RBCK1

![GO enrichment restricted to the immune-flagged subset](https://raw.githubusercontent.com/https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb/main/results/enrichr_immune_subset.png)

*(This figure illustrates the specific immune processes represented within the confirmed subset: immune effector process, immune response-regulating signaling pathway, glial cell proliferation, microglial cell proliferation, regulation of immune response, positive regulation of phagocytosis/engulfment, and adaptive immune response.)*

### 5.3 STRING network analysis
![STRING protein-protein interaction network, full 76-gene list](https://raw.githubusercontent.com/https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb/main/results/string_network.png)

Submitting the full, unbiased 76-gene list to STRING produced a network in which immune-flagged genes (TREM2, SPI1, PLCG2, BLNK, INPP5D, TREML2) formed a distinct, densely interconnected cluster, visually and topologically separable from a second cluster of amyloid-processing genes (APP, SORL1, CLU, APH1B, ABI3, CTSB). This clustering emerged from documented protein interaction evidence alone, independent of the GO-term-based enrichment analysis above — providing a second, methodologically distinct line of evidence for the same conclusion.

![STRING functional enrichment table](https://raw.githubusercontent.com/https://github.com/Harshika0501/Alzheimers-Disease-Immune-Pathways/edit/main/AD_Immune_Pathways_Notebook.ipynb/main/results/string_enrichment_table.png)

STRING's own enrichment output independently confirmed "Immune effector process" (GO:0002252) as significantly enriched (13 of 76 genes, FDR = 1.34 × 10⁻⁵), along with related terms including "Myeloid leukocyte activation" (FDR = 1.2 × 10⁻⁴), "Positive regulation of immune system process" (FDR = 1.60 × 10⁻⁶), "Regulation of immune response" (FDR = 1.34 × 10⁻⁵), "Macrophage activation" (FDR = 1.4 × 10⁻³), and "Microglial cell activation" (FDR = 1.9 × 10⁻³).

---
## 6. Discussion

### 6.1 What this analysis shows, and why it matters
The question this project was built to answer is narrow and falsifiable: does a reanalysis run independently — different tools, different statistical backend, different analyst — recover the same immune signal Bellenguez et al. (2022) reported, or was that signal an artifact of their specific pipeline? Three methodologically distinct checks (Enrichr across GO/KEGG/Reactome, STRING network topology, and STRING's own independent enrichment engine) all converge on the same answer: yes. That convergence, more than any single gene's individual role, is the actual finding here — it's evidence that the immune signal in AD genetics is a comparatively stable feature of the data, not a byproduct of one group's analytical choices.

That matters because AD genetics is usually discussed almost entirely in amyloid/tau terms. Here, a comparable fraction of the genome-wide significant risk genes — 26 of 76, roughly a third — cluster around innate immune and microglial function rather than amyloid processing itself, and they form a topologically distinct network module rather than scattering randomly among the amyloid-associated genes. The implication is that at least part of inherited AD risk operates through how the brain's immune system manages debris, complement tagging, and synaptic material — a mechanism that is upstream of, and partly independent from, amyloid production or aggregation. That reframing is what motivates immune-directed and complement-directed therapeutic strategies as a parallel track to amyloid-clearing ones, rather than as a secondary afterthought to them.

### 6.2 Mechanistic anchors for the statistical signal
A few of the flagged genes give the enrichment result a concrete biological story rather than leaving it as an abstract p-value. TREM2 — the largest, most connected node in the immune cluster — sits at the center of the microglial "disease-associated microglia" (DAM) switch that governs amyloid clearance, and its risk signal is among the strongest non-APOE effects in AD GWAS, which is consistent with its topological centrality here. CR1 and CLU point to the complement-mediated synaptic pruning pathway, the same developmental mechanism Stevens, Hong, and colleagues showed is reactivated in AD and correlates with the synapse loss that tracks cognitive decline more tightly than plaque burden does. HLA-DQA1 is the one gene in the set that implicates the *adaptive* rather than innate immune arm, suggesting the immune contribution isn't limited to microglial housekeeping. And SHARPIN, RBCK1, TNIP1, ADAM17, and SPPL2A converge on TNF-α/LUBAC signaling — a pathway the original authors also flagged as a genuinely novel finding, not a known one being re-confirmed, which makes its independent recovery here a stronger check than re-finding an already-obvious signal would have been.

### 6.3 Why this is relevant to immunotherapy and vaccinology
This isn't a purely descriptive result — it sits directly upstream of an active area of clinical development. Amyloid immunotherapy has a two-decade track record, from the halted AN1792 vaccine trial (stopped after some patients developed T-cell-mediated meningoencephalitis, though responders showed genuine plaque clearance) through today's approved passive antibodies like lecanemab and donanemab. Both the therapeutic effect and the main safety concern of that entire drug class — ARIA, itself thought to involve vascular immune/inflammatory responses — are inseparable from the same innate-immune biology this analysis independently recovered. In other words, the genetic evidence here isn't just consistent with immunotherapy as a strategy; it's a partial explanation for why manipulating the immune system, not only the amyloid protein itself, has clinical consequences in this disease.

### 6.4 Limitations
- Keyword-based term screening is not a formal traversal of the GO ontology's hierarchical structure; genes could be missed if their associated term names lack an exact keyword match, and this analysis does not use tools such as QuickGO/AmiGO that would allow a structurally rigorous check against the formal GO:0002376 ("immune system process") subtree.
- Enrichment analysis identifies statistical over-representation of annotated gene function; it does not constitute experimental validation that these genes causally act through immune mechanisms specifically in the context of AD pathology.

---
## 7. Scope

This is a **beginner-level, self-directed learning project**, undertaken to build practical skills in gene set enrichment analysis and protein interaction network analysis, and to explore a specific scientific question.

The objective was to **independently attempt to recreate, using freely available bioinformatics tools, a finding already reported in a published, peer-reviewed GWAS meta-analysis** (Bellenguez et al., 2022) — specifically, their finding that immune system processes are significantly enriched among Alzheimer's disease (AD) genetic risk genes, alongside the expected amyloid/tau pathway signal. All analyses were performed on a gene list derived directly from the published paper, using standard, publicly available web tools (Enrichr, STRING).

---
## 8. Tools used

- Python (pandas) — data loading
- [Enrichr](https://maayanlab.cloud/Enrichr/) — gene set enrichment analysis
- [STRING](https://string-db.org/) — protein–protein interaction network analysis
- [GWAS Catalog](https://www.ebi.ac.uk/gwas/) — exploratory reference only, not used for the final gene list

---
## 9. References

1. Bellenguez, C., Küçükali, F., Jansen, I.E. et al. (2022). New insights into the genetic etiology of Alzheimer's disease and related dementias. *Nature Genetics*, 54, 412–436. https://doi.org/10.1038/s41588-022-01024-z
2. Ulland, T.K. & Colonna, M. (2018). TREM2 — a key player in microglial biology and Alzheimer disease. *Nature Reviews Neurology*, 14, 667–675. https://doi.org/10.1038/s41582-018-0072-1
3. Heppner, F.L., Ransohoff, R.M., & Becher, B. (2015). Immune attack: the role of inflammation in Alzheimer disease. *Nature Reviews Neuroscience*, 16, 358–372. https://doi.org/10.1038/nrn3880